# Análisis de sentimiento de reseñas con redes recurrentes (LSTM vs GRU)

**Objetivo:** clasificar reseñas de películas como *positivas* o *negativas* usando una red recurrente bidireccional, comparar LSTM contra GRU y dejar el modelo listo para una app Streamlit.

> **Colab:** activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

In [ ]:
import os, sys, json, re
REPO_URL = "https://github.com/SantiagoPerdomo25/rnn-sentimiento"   # <-- CAMBIAR si tu repo se llama distinto
if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone {REPO_URL} repo
    %cd repo
if os.path.basename(os.getcwd()) == "notebooks":   # ejecución local desde /notebooks
    os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from src import data, model as model_lib, evaluate

tf.keras.utils.set_random_seed(data.SEED)
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Selección y descripción del dataset

Se usa **IMDB Large Movie Review Dataset** (Maas et al., Stanford): **50.000 reseñas** de películas en inglés, perfectamente balanceadas (25.000 positivas y 25.000 negativas). Las reseñas con puntuación ≤ 4/10 son negativas y las ≥ 7/10 positivas; las neutras se excluyeron.

**Por qué este dataset:**
- Es un problema de **datos secuenciales** (texto): el orden de las palabras y el contexto importan ("not good" ≠ "good").
- Las reseñas son largas y variables, lo que justifica el uso de memoria (LSTM/GRU).
- Es un benchmark conocido, con tamaño suficiente para entrenar en Colab.

In [ ]:
data_dir = data.download_dataset("data")
df = data.load_dataframe(data_dir)       # lee train + test originales y los junta (50.000)
print("Forma:", df.shape)
display(df.head())
display(df.groupby(["original_split", "sentimiento"]).size().unstack())

## 2. Análisis exploratorio de los datos (EDA)

In [ ]:
# Balance de clases
df["sentimiento"].value_counts().plot(kind="bar", color=["tomato", "seagreen"], rot=0,
                                      title="Reseñas por sentimiento")
plt.ylabel("Cantidad"); plt.show()
print("Nulos:", df["text"].isna().sum(), "| Duplicados:", df["text"].duplicated().sum())

In [ ]:
# Longitud de las reseñas (en palabras)
df["n_palabras"] = df["text"].str.split().str.len()
display(df.groupby("sentimiento")["n_palabras"].describe().round(1))
print("Percentiles globales:", df["n_palabras"].quantile([.5, .75, .9, .95]).to_dict())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for s, c in zip(data.CLASS_NAMES, ["tomato", "seagreen"]):
    ax[0].hist(df.loc[df.sentimiento == s, "n_palabras"].clip(upper=1000), bins=50, alpha=.6, label=s, color=c)
ax[0].axvline(model_lib.SEQ_LEN, color="k", ls="--", label=f"SEQ_LEN={model_lib.SEQ_LEN}")
ax[0].set(title="Distribución de longitud (recortada a 1000)", xlabel="Palabras"); ax[0].legend()
df.boxplot(column="n_palabras", by="sentimiento", ax=ax[1], showfliers=False)
ax[1].set(title="Longitud por sentimiento", xlabel="", ylabel="Palabras"); plt.suptitle("")
plt.tight_layout(); plt.show()
cobertura = (df["n_palabras"] <= model_lib.SEQ_LEN).mean()
print(f"Con SEQ_LEN={model_lib.SEQ_LEN}, el {cobertura:.0%} de las reseñas cabe completo; el resto se recorta.")

In [ ]:
# Palabras más frecuentes por clase (sin stopwords)
stop = set(ENGLISH_STOP_WORDS) | {"br", "movie", "film", "s", "t"}
def top_words(texts, n=15):
    c = Counter(w for t in texts for w in re.findall(r"[a-z']+", t.lower()) if w not in stop and len(w) > 2)
    return c.most_common(n)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, s, col in zip(ax, data.CLASS_NAMES, ["tomato", "seagreen"]):
    words, counts = zip(*top_words(df.loc[df.sentimiento == s, "text"]))
    a.barh(words[::-1], counts[::-1], color=col); a.set_title(f"Top palabras: {s}")
plt.tight_layout(); plt.show()

In [ ]:
# Ejemplos reales
for s in data.CLASS_NAMES:
    print(f"--- {s.upper()} ---")
    for t in df[df.sentimiento == s]["text"].sample(2, random_state=1):
        print("•", t[:300].replace("\n", " "), "...\n")

## 3. Limpieza y preprocesamiento

1. **Duplicados:** se eliminan *antes* de dividir, para que una misma reseña no quede a la vez en entrenamiento y prueba (fuga de datos).
2. **Limpieza de texto:** se eliminan etiquetas HTML (`<br />`), URLs y espacios repetidos.
3. **Tokenización y vectorización:** una capa `TextVectorization` (minúsculas, sin puntuación) convierte cada reseña en una secuencia de **300 enteros** (vocabulario de 20.000 palabras; relleno o recorte según haga falta).
4. La capa va **dentro del modelo**, así la app Streamlit recibe texto crudo y no hay desajuste entre entrenamiento e inferencia.

In [ ]:
ejemplo = df["text"].iloc[3]
print("ANTES:  ", ejemplo[:300])
print("DESPUÉS:", data.clean_text(ejemplo)[:300])

df = df.drop_duplicates(subset="text").reset_index(drop=True)
df["text_clean"] = df["text"].map(data.clean_text)
print("\nReseñas tras eliminar duplicados:", len(df))

## 4. División en entrenamiento, validación y prueba

División **estratificada 70 / 15 / 15** con semilla fija. El conjunto de prueba no se usa hasta la evaluación final.

In [ ]:
splits = data.split_data(df)
display(pd.DataFrame({k: {"total": len(v), "positivas": int(v.label.sum()),
                          "negativas": int((1 - v.label).sum())} for k, v in splits.items()}).T)

ds = {k: data.make_dataset(v["text_clean"], v["label"], batch_size=64, training=(k == "train"))
      for k, v in splits.items()}

In [ ]:
# El vocabulario se construye SOLO con entrenamiento (evita fuga de información)
vectorizer = model_lib.build_vectorizer(splits["train"]["text_clean"])
vocab = vectorizer.get_vocabulary()
print("Tamaño del vocabulario:", len(vocab), "| primeras palabras:", vocab[:15])

frase = "This movie was absolutely wonderful"
print("\nTexto:", frase)
print("Secuencia:", vectorizer([frase]).numpy()[0][:12], "...")

## 5. Construcción de la red neuronal

Arquitectura: *TextVectorization → Embedding(64, con máscara) → **Bidirectional(LSTM o GRU, 64)** → Dropout(0.5) → Dense(64) → Dropout(0.3) → Sigmoide*.

- **Embedding:** representa cada palabra como un vector denso aprendido.
- **Bidireccional:** lee la reseña de inicio a fin y de fin a inicio para captar contexto en ambos sentidos.
- **Dropout:** reduce el sobreajuste.
- Se entrenan **dos variantes (LSTM y GRU)** y se conserva la mejor según la validación.

In [ ]:
model_lstm = model_lib.build_model(vectorizer, "lstm")
model_lstm.summary()

## 6. Entrenamiento y comparación LSTM vs GRU

In [ ]:
os.makedirs("models", exist_ok=True); os.makedirs("reports", exist_ok=True)
models, histories, rows = {}, {}, []
for cell in ["lstm", "gru"]:
    print(f"\n===== Entrenando {cell.upper()} =====")
    tf.keras.utils.set_random_seed(data.SEED)
    m = model_lib.build_model(vectorizer, cell)
    h = m.fit(ds["train"], validation_data=ds["val"], epochs=8, verbose=2,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                                          restore_best_weights=True)])
    val_loss, val_acc, val_auc = m.evaluate(ds["val"], verbose=0)
    models[cell], histories[cell] = m, h
    rows.append({"modelo": cell.upper(), "épocas": len(h.history["loss"]), "val_loss": val_loss,
                 "val_accuracy": val_acc, "val_auc": val_auc})

comparacion = pd.DataFrame(rows).set_index("modelo").round(4)
display(comparacion)
best = comparacion["val_accuracy"].idxmax().lower()
model = models[best]
print("Mejor modelo en validación:", best.upper())

## 7. Gráficas del entrenamiento

In [ ]:
evaluate.plot_history(histories[best], "reports/training_curves.png")
plt.suptitle(f"Modelo final: {best.upper()}", y=1.02); plt.show()

# Comparación de ambas variantes
plt.figure(figsize=(6, 4))
for cell, h in histories.items():
    plt.plot(h.history["val_accuracy"], marker="o", label=f"{cell.upper()} (val)")
plt.title("Accuracy de validación: LSTM vs GRU"); plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.legend()
plt.tight_layout(); plt.savefig("reports/comparacion_lstm_gru.png", dpi=150); plt.show()

## 8. Evaluación en el conjunto de prueba

El dataset está balanceado, pero no basta el accuracy: se reportan también **precision, recall, F1 y ROC-AUC**, además de la **matriz de confusión** y la **curva ROC**.

In [ ]:
y_true, probs, y_pred = evaluate.get_predictions(model, ds["test"])
metricas = evaluate.metrics_dict(y_true, probs, y_pred)
display(pd.Series(metricas, name="test").round(4).to_frame())
print(evaluate.text_report(y_true, y_pred))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=data.CLASS_NAMES).plot(ax=ax[0], cmap="Blues")
ax[0].set_title("Matriz de confusión (prueba)")
fpr, tpr, _ = roc_curve(y_true, probs)
ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_true, probs):.3f}"); ax[1].plot([0, 1], [0, 1], "--", color="gray")
ax[1].set(title="Curva ROC", xlabel="Falsos positivos", ylabel="Verdaderos positivos"); ax[1].legend()
plt.tight_layout(); plt.show()
evaluate.plot_confusion_matrix(y_true, y_pred, "reports/confusion_matrix.png"); plt.close()
evaluate.plot_roc(y_true, probs, "reports/roc_curve.png"); plt.close()

## 9. Predicciones de prueba (12 reseñas del conjunto de prueba)

In [ ]:
tabla = evaluate.prediction_table(model, splits["test"], n=12)
pd.set_option("display.max_colwidth", 120)
display(tabla)
tabla.to_csv("reports/predictions.csv", index=False, encoding="utf-8")
print("Aciertos:", (tabla["acierto"] == "✅").sum(), "de", len(tabla))

In [ ]:
# Errores más "seguros" del modelo (ayudan a entender sus límites)
test = splits["test"].copy()
test["prob"] = probs; test["pred"] = y_pred
mal = test[test.pred != test.label].copy()
mal["confianza"] = np.where(mal.pred == 1, mal.prob, 1 - mal.prob)
for _, r in mal.sort_values("confianza", ascending=False).head(4).iterrows():
    print(f"Real: {data.CLASS_NAMES[r.label]} | Pred: {data.CLASS_NAMES[r.pred]} ({r.confianza:.0%})")
    print("  ", r.text_clean[:250], "...\n")

In [ ]:
# Prueba con frases nuevas (incluye casos difíciles: negación y sarcasmo)
from src.predict import predict_texts
frases = [
    "I loved every minute of this movie, a true masterpiece!",
    "Terrible acting and a boring script. Do not waste your money.",
    "It was not good at all.",
    "I expected much more, but it was fine I guess.",
    "Oh great, another two hours of my life I will never get back.",
]
for f, r in zip(frases, predict_texts(model, frases)):
    print(f"{r['label']:>8} ({r['confidence']:.0%}) <- {f}")

## 10. Guardado del modelo

In [ ]:
model.save("models/sentimiento_rnn.keras")
with open("reports/history.json", "w") as f:
    json.dump(histories[best].history, f)
with open("reports/metrics.json", "w") as f:
    json.dump({**metricas, "modelo": best}, f, indent=2)
with open("reports/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(evaluate.text_report(y_true, y_pred))
comparacion.to_csv("reports/comparacion_modelos.csv")

# Comprobación: recargar y predecir como lo hará Streamlit
from src.predict import load_model, predict_text
m2 = load_model()
print(predict_text(m2, "A wonderful and moving film."))

In [ ]:
# En Colab: descarga los artefactos para subirlos al repositorio
if "google.colab" in sys.modules:
    !zip -r artefactos.zip models reports
    from google.colab import files
    files.download("artefactos.zip")

## 11. Conclusiones

- Indica el accuracy, F1 y AUC obtenidos en prueba y qué modelo ganó (LSTM o GRU) y por cuánto.
- Comenta qué tipo de errores comete (negación, sarcasmo, reseñas mixtas o muy largas recortadas a 300 palabras).
- Menciona limitaciones: solo funciona en inglés, el vocabulario es de 20.000 palabras y reseñas de otros dominios pueden fallar.
- Posibles mejoras: embeddings preentrenados (GloVe), mayor longitud de secuencia, atención o modelos Transformer.